# 01_qc — SnapATAC2 pipeline

**Dataset:** 10x Genomics ATACv2 PBMC 10k (hg38, Cellranger-ATAC v2, March 2022)
**Parallel to:** `../signac_pbmc_10k/01_qc.R` (Signac R workflow)

Six blocks: setup → load fragments → QC metrics → filter → dim reduction + clustering → annotation.

In [ ]:
# Block 1 - Setup: imports, seed, paths, sanity check

import snapatac2 as snap
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import logging

# Silence plotly export logs
logging.getLogger("choreographer").setLevel(logging.WARNING)
logging.getLogger("kaleido").setLevel(logging.WARNING)

np.random.seed(42)

# Data paths (peak matrix + metadata on Windows disk, fragments on WSL native fs)
data_dir = Path("/mnt/d/scATAC-seq/data/pbmc_10k_atacv2")
sample = "10k_pbmc_ATACv2_nextgem_Chromium_Controller"

counts_path   = data_dir / f"{sample}_filtered_peak_bc_matrix.h5"
metadata_path = data_dir / f"{sample}_singlecell.csv"

# Fragments in WSL native (~10x faster random I/O than /mnt/d/ for hdf5)
frag_path = Path.home() / "scatac_data" / f"{sample}_fragments.tsv.gz"

# Output paths
Path("data").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)
output_h5ad = str(Path.home() / "scatac_data" / "pbmc_10k.h5ad")

# Sanity check
assert counts_path.exists(),   f"Missing: {counts_path}"
assert metadata_path.exists(), f"Missing: {metadata_path}"
assert frag_path.exists(),     f"Missing: {frag_path}"

print("Block 1 complete. Paths verified, output dirs ready.")

In [ ]:
# Block 2 - Load fragments into backed AnnData
# snap.genome.hg38 has chrom_sizes hardcoded (no fasta download needed).
# Backed h5ad written to WSL native fs to avoid cross-fs random write deadlock.

import os
if os.path.exists(output_h5ad):
    os.remove(output_h5ad)

data = snap.pp.import_fragments(
    fragment_file=str(frag_path),
    chrom_sizes=snap.genome.hg38,
    file=output_h5ad,
    sorted_by_barcode=False,
    min_num_fragments=200,
)

data

In [ ]:
# Block 3 - Compute per-cell QC metrics
# Uses built-in hg38 annotation for TSS enrichment.

snap.metrics.tsse(data, snap.genome.hg38)
snap.metrics.frag_size_distr(data)

# Preview obs (snapatac2 backed obs is a Polars DataFrame accessed via [:])
obs = data.obs[:]
print("Available obs columns:", obs.columns)
print("
Distribution summary:")
print(obs.describe())
print("
First 5 cells:")
print(obs.head())

In [ ]:
# Block 4 - QC visualization + cell filtering

# QC scatter (log fragment count vs TSS enrichment)
snap.pl.tsse(data, interactive=False, out_file="results/qc_tsse_scatter.png")

# Fragment size distribution (nucleosome pattern)
snap.pl.frag_size_distr(data, interactive=False, out_file="results/qc_frag_size.png")

# Filter cells
n_before = data.n_obs
snap.pp.filter_cells(
    data,
    min_counts=3000,
    min_tsse=2,
    max_counts=100000,
)
n_after = data.n_obs
print(f"Cells before: {n_before}, after: {n_after} (kept {100 * n_after / n_before:.1f}%)")

print("
Post-filter obs summary:")
print(data.obs[:].describe())

In [ ]:
# Block 5 - Tile matrix + spectral + clustering + UMAP

# 1. Tile matrix (cell x 500bp genomic bins) — analog of peak matrix in Signac
snap.pp.add_tile_matrix(data, bin_size=500)
print(data)

# 2. Feature selection (top variable tiles)
snap.pp.select_features(data, n_features=25000)
print(f"Selected {sum(data.var['selected'])} of {data.n_vars} features")

# 3. Spectral embedding (LSI equivalent; handles depth confound internally)
snap.tl.spectral(data)
print(f"Spectral embedding shape: {data.obsm['X_spectral'].shape}")

# 4. KNN + Leiden clustering
snap.pp.knn(data)
snap.tl.leiden(data, resolution=0.8)
n_clusters = len(set(data.obs["leiden"]))
print(f"Found {n_clusters} clusters")

# 5. UMAP layout
snap.tl.umap(data)

# 6. Plot UMAP with cluster colors
snap.pl.umap(data, color="leiden", interactive=False, out_file="results/umap_clusters.png")
print("UMAP saved to results/umap_clusters.png")

In [ ]:
# Block 6 - Cell type annotation via gene activity matrix

# Build gene matrix (analog of Signac GeneActivity)
gene_matrix = snap.pp.make_gene_matrix(data, gene_anno=snap.genome.hg38, inplace=False)

# Copy UMAP coords + cluster labels so scanpy can plot
gene_matrix.obsm["X_umap"] = data.obsm["X_umap"]
gene_matrix.obs["leiden"] = data.obs["leiden"]

# Normalize + log-transform (equivalent of Signac NormalizeData)
sc.pp.normalize_total(gene_matrix, target_sum=1e4)
sc.pp.log1p(gene_matrix)

# Canonical PBMC marker genes
marker_genes = ["MS4A1", "CD3E", "CD8A", "CD4", "NKG7", "CD14", "LYZ", "FCGR3A"]
marker_genes = [g for g in marker_genes if g in gene_matrix.var_names]
print(f"Plotting {len(marker_genes)} markers")

# Plot with scanpy
sc.set_figure_params(dpi=100)
sc.pl.umap(
    gene_matrix,
    color=marker_genes,
    ncols=4,
    vmin=0,
    vmax="p99",
    save="_markers_normalized.png",
)